# GDAS latency on NODD

Reproduces the latency numbers in [`gdas-support.md`](../gdas-support.md) from the public NOAA
Open Data Dissemination bucket (`s3://noaa-gfs-bdp-pds`), read anonymously:

1. How long after its cycle each GDAS and GFS 0-hour file is published.
2. Which days are missing from the archive.
3. How long a granule waits for its inputs, by longitude.
4. The wait after a Sentinel-2 granule enters our bucket, waiting for the right file versus the
   next file to land.
5. The same with GFS.

Run from the repository root with `uv run --with jupyterlab jupyter lab`. The dates are fixed in
the next cell, so a rerun gives the same numbers, except for any file the bucket rewrites later
(`LastModified` is the time of the last write).

In [1]:
import datetime as dt
from concurrent.futures import ThreadPoolExecutor

import boto3
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError
from lasrc_ancillary.providers.gdas import NODD_BUCKET, find_gdas, gdas_cycles

from lasrc_ancillary import estimate_acquisition_time

# Publish delays: every cycle in the year before END.
END = dt.datetime(2026, 11, 16, tzinfo=dt.UTC)
DAYS = 365

# Archive coverage: the 12Z cycle of every day in this range.
COVERAGE_START = dt.date(2013, 4, 1)
COVERAGE_END = dt.date(2026, 11, 15)

# Sentinel-2 L1C sensing to entering our bucket (median, May to September 2026): 3.22 h of ESA
# processing and 1.12 h of download. HLS processing after the inputs are ready takes about 0.6 h.
ENTRY_DELAY = dt.timedelta(hours=3.22 + 1.12)
COMPUTE = dt.timedelta(hours=0.6)

s3 = boto3.client(
    "s3",
    config=Config(
        signature_version=UNSIGNED,
        max_pool_connections=24,
        retries={"mode": "standard", "total_max_attempts": 10},
    ),
)
pd.set_option("display.precision", 2)

## 1. Publish delay

Hours from each cycle to the `LastModified` time of its 0-hour file, for GDAS and GFS at 1 and
0.25 degree. Every cycle in the period uses the `atmos/` key layout.

In [2]:
def key(system, resolution, cycle):
    hh = f"{cycle:%H}"
    return f"{system}.{cycle:%Y%m%d}/{hh}/atmos/{system}.t{hh}z.pgrb2.{resolution}.f000"


def published(job):
    system, resolution, cycle = job
    try:
        head = s3.head_object(Bucket=NODD_BUCKET, Key=key(system, resolution, cycle))
    except ClientError:
        return (*job, None)
    return (*job, head["LastModified"])


start = END - dt.timedelta(days=DAYS)
cycles = [start + dt.timedelta(hours=6 * i) for i in range(DAYS * 4)]
jobs = [(s, r, c) for s in ("gdas", "gfs") for r in ("1p00", "0p25") for c in cycles]
with ThreadPoolExecutor(24) as pool:
    pub = pd.DataFrame(
        list(pool.map(published, jobs)),
        columns=["system", "resolution", "cycle", "published"],
    )
pub["cycle"] = pd.to_datetime(pub["cycle"], utc=True)
pub["published"] = pd.to_datetime(pub["published"], utc=True)
pub["hours"] = (pub["published"] - pub["cycle"]).dt.total_seconds() / 3600
print("missing files:", pub["published"].isna().sum(), "of", len(pub))
pub.groupby(["system", "resolution"])["hours"].describe(percentiles=[0.5, 0.95, 0.99])

missing files: 0 of 5840


count  mean   std   min   50%   95%   99%    max
system resolution                                                   
gdas   0p25        1460.0  6.96  0.29  6.73  6.85  7.61  7.92   9.73
       1p00        1460.0  6.96  0.29  6.72  6.84  7.60  7.98   9.73
gfs    0p25        1460.0  3.63  0.30  3.48  3.57  3.81  3.85  13.45
       1p00        1460.0  3.61  0.16  3.48  3.56  3.80  3.83   7.46

By cycle hour, 1 degree. The GDAS 18Z cycle is the slowest.

In [3]:
one_degree = pub[pub["resolution"] == "1p00"]
(
    one_degree.groupby(["system", one_degree["cycle"].dt.hour])["hours"]
    .quantile([0.05, 0.5, 0.95])
    .unstack()
)

0.05  0.50  0.95
system cycle                  
gdas   0      6.89  6.94  7.17
       6      6.75  6.79  6.85
       12     6.76  6.81  7.02
       18     6.80  7.21  7.85
gfs    0      3.52  3.58  3.82
       6      3.50  3.54  3.78
       12     3.51  3.55  3.79
       18     3.53  3.58  3.80

GDAS 1 degree by month, to show drift.

In [4]:
gdas = pub[(pub["system"] == "gdas") & (pub["resolution"] == "1p00")]
(
    gdas.groupby(gdas["cycle"].dt.strftime("%Y-%m"))["hours"]
    .quantile([0.5, 0.95])
    .unstack()
)

,0.50,0.95
cycle,,
2025-11,6.81,7.15
2025-12,6.80,7.14
2026-01,6.80,7.06
2026-02,6.81,7.32
2026-03,6.82,7.74
2026-04,6.89,7.74
2026-05,6.86,7.70
2026-06,6.90,7.79
2026-07,6.94,7.66


## 2. Archive coverage

Whether the 12Z 1 degree 0-hour file exists for each day, using the same lookup as
`lasrc-ancillary` (all four key layouts). This takes a minute or two.

In [5]:
def has_12z(day):
    cycle = dt.datetime(day.year, day.month, day.day, 12, tzinfo=dt.UTC)
    return day, find_gdas([cycle], s3) is not None


days = [d.date() for d in pd.date_range(COVERAGE_START, COVERAGE_END, freq="D")]
with ThreadPoolExecutor(24) as pool:
    coverage = pd.DataFrame(list(pool.map(has_12z, days)), columns=["day", "present"])
coverage["year"] = [d.year for d in coverage["day"]]
coverage.groupby("year")["present"].agg(days="size", missing=lambda s: int((~s).sum()))

,days,missing
year,,
2013,275,0
2014,365,2
2015,365,17
2016,366,16
2017,365,1
2018,365,34
2019,365,59
2020,366,0
2021,365,0


Missing days, grouped into date ranges.

In [6]:
missing = coverage[~coverage["present"]].reset_index(drop=True)
run = (pd.Series(missing["day"]).diff() != dt.timedelta(days=1)).cumsum()
gaps = missing.groupby(run)["day"].agg(first="min", last="max", days="size")
gaps.reset_index(drop=True)

,first,last,days
0,2014-07-04,2014-07-04,1
1,2014-09-13,2014-09-13,1
2,2015-01-21,2015-01-21,1
3,2015-01-30,2015-01-30,1
4,2015-02-06,2015-02-06,1
5,2015-02-23,2015-02-24,2
6,2015-04-09,2015-04-09,1
7,2015-04-12,2015-04-12,1
8,2015-06-16,2015-06-16,1
9,2015-07-23,2015-07-23,1


## 3. Wait after acquisition, by longitude

A granule needs the cycle after its acquisition (`cycle2`), which lands at `cycle2` plus the
publish delay. This uses the median GDAS 1 degree delay from section 1, the Sentinel-2 orbit and
the time estimate in `lasrc_ancillary`, at the equator.

In [7]:
gdas_delay = dt.timedelta(hours=gdas["hours"].median())
DAY = dt.date(2026, 3, 14)


def wait_after_acquisition(lat, lon, delay):
    t = estimate_acquisition_time("SENTINEL_2", DAY, lat, lon)
    later = gdas_cycles(t)[1]
    return t, later, (later + delay - t).total_seconds() / 3600


rows = []
for lon in range(-180, 180, 15):
    t, later, wait = wait_after_acquisition(0.0, lon, gdas_delay)
    rows.append((lon, f"{t:%H:%M}", f"{later:%d %HZ}", wait))
pd.DataFrame(rows, columns=["lon", "utc", "cycle2", "wait_h"])

,lon,utc,cycle2,wait_h
0,-180,22:30,15 00Z,8.34
1,-165,21:30,15 00Z,9.34
2,-150,20:30,15 00Z,10.34
3,-135,19:30,15 00Z,11.34
4,-120,18:30,15 00Z,12.34
5,-105,17:30,14 18Z,7.34
6,-90,16:30,14 18Z,8.34
7,-75,15:30,14 18Z,9.34
8,-60,14:30,14 18Z,10.34
9,-45,13:30,14 18Z,11.34


Some capitals, at their real latitude.

In [8]:
capitals = {
    "Mexico City": (19.4, -99.1),
    "Washington": (38.9, -77.0),
    "Brasilia": (-15.8, -47.9),
    "London": (51.5, -0.1),
    "Nairobi": (-1.3, 36.8),
    "Tehran": (35.7, 51.4),
    "New Delhi": (28.6, 77.2),
    "Tokyo": (35.7, 139.7),
    "Canberra": (-35.3, 149.1),
    "Wellington": (-41.3, 174.8),
}
rows = []
for name, (lat, lon) in capitals.items():
    t, later, wait = wait_after_acquisition(lat, lon, gdas_delay)
    rows.append((name, f"{t:%H:%M}", f"{later:%d %HZ}", wait))
pd.DataFrame(rows, columns=["capital", "utc", "cycle2", "wait_h"])

,capital,utc,cycle2,wait_h
0,Mexico City,17:18,14 18Z,7.53
1,Washington,16:06,14 18Z,8.74
2,Brasilia,13:31,14 18Z,11.31
3,London,11:14,14 12Z,7.60
4,Nairobi,08:02,14 12Z,10.81
5,Tehran,07:29,14 12Z,11.35
6,New Delhi,05:40,14 06Z,7.17
7,Tokyo,01:36,14 06Z,11.24
8,Canberra,00:08,14 06Z,12.69
9,Wellington,22:20,15 00Z,8.51


## 4. Wait after entering our bucket

Replays acquisitions every 10 minutes through the period against the real publish times from
section 1. Each granule enters our bucket `ENTRY_DELAY` after acquisition. Two waits are compared:

- **cycle2**: until the file for the cycle after acquisition lands. This is what the granule
  needs.
- **next file**: until whichever GDAS file lands next after entry. This is what the 2026 latency
  study measured. Often it is the cycle before acquisition.

Acquisition times are spread evenly through the day here. Real granules are not spread evenly in
longitude, so the real shares depend on where the tiles are.

In [9]:
def replay(published_times):
    landed = published_times.dropna().sort_index()
    t = pd.date_range(landed.index[1], landed.index[-2], freq="10min", inclusive="left")
    entry = pd.Series(t + ENTRY_DELAY)
    # The later of the two cycles that bracket t (the cycle rule in gdas_cycles).
    needed = landed.reindex(t.floor("6h") + pd.Timedelta(hours=6)).reset_index(
        drop=True
    )
    landings = landed.sort_values().reset_index(drop=True)
    first_after_entry = landings.searchsorted(entry)
    keep = (first_after_entry < len(landings)) & needed.notna().to_numpy()
    entry, needed = entry[keep], needed[keep]
    next_landing = landings.iloc[first_after_entry[keep]].set_axis(entry.index)
    hour = pd.Timedelta(hours=1)
    return pd.DataFrame(
        {
            "cycle2_h": (needed - entry).clip(lower=pd.Timedelta(0)) / hour,
            "next_file_h": (next_landing - entry) / hour,
        }
    ).reset_index(drop=True)


def summarize(waits):
    limit_h = 6 - COMPUTE.total_seconds() / 3600
    return pd.DataFrame(
        {
            "median_wait_h": waits.median(),
            "published_within_6h_of_entry": (waits <= limit_h).mean(),
        }
    )


gdas_waits = replay(gdas.set_index("cycle")["published"])
summarize(gdas_waits)

,median_wait_h,published_within_6h_of_entry
cycle2_h,5.72,0.45
next_file_h,3.00,0.89


## 5. GFS

The same replay with GFS 0-hour files, which come from the same model with an earlier
observation cutoff.

In [10]:
gfs = pub[(pub["system"] == "gfs") & (pub["resolution"] == "1p00")]
summarize(replay(gfs.set_index("cycle")["published"]))

,median_wait_h,published_within_6h_of_entry
cycle2_h,2.36,0.99
next_file_h,3.00,0.90
